# Copia del notebook del productor modificado para el ejercicio 2.7


In [26]:
from kafka import KafkaConsumer
import json
import matplotlib.pyplot as plt
from collections import deque
from datetime import datetime

In [27]:
# Configuracion global

MAX_SAMPLES = 100
OUTPUT_PATH = "/opt/local/water_quality_plot_many_copy.png"
METRICS = {
    "water_temperature": {"title": "Temp", "unit": "°C", "color": "blue"},
    "ph_level":          {"title": "pH",   "unit": "pH", "color": "green"},
    "turbidity":         {"title": "Turb", "unit": "NTU", "color": "orange"},
    "dissolved_oxygen":  {"title": "DO",   "unit": "mg/L", "color": "red"}
}

In [28]:
# Almacenamiento dinámico
data_by_sensor = {}

In [29]:
def render_plots():
    """Genera y guarda la visualización con formato de tiempo correcto."""
    sensor_ids = sorted(data_by_sensor.keys())
    num_sensors = len(sensor_ids)
    if num_sensors == 0: return

    fig, axes = plt.subplots(
        nrows=num_sensors, 
        ncols=len(METRICS), 
        figsize=(16, 4 * num_sensors),
        squeeze=False
    )

    for i, s_id in enumerate(sensor_ids):
        storage = data_by_sensor[s_id]
        
        # Convertimos los timestamps (float/int) a objetos datetime de Python
        # para que Matplotlib entienda que es tiempo
        times = [datetime.fromtimestamp(t) for t in storage["timestamps"]]
        
        for j, (key, config) in enumerate(METRICS.items()):
            ax = axes[i, j]
            ax.plot(times, list(storage[key]), color=config["color"], linewidth=1.5)
            
            # --- MEJORAS DE FORMATO ---
            ax.set_title(f"Sensor {s_id} | {config['title']}")
            ax.set_ylabel(config["unit"])
            
            # Formatear el eje X para mostrar HH:MM:SS
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
            
            # Rotar las etiquetas del tiempo para que no se solapen
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

    plt.tight_layout(pad=3.0)
    plt.savefig(OUTPUT_PATH)
    plt.close(fig)

In [30]:
def run_consumer(consumer):
    """Bucle principal de consumo."""
    print("Iniciando consumo de mensajes...")
    try:
        for message in consumer:
            sensor_id = process_message(message.value)
            print(f"Sensor ID: {sensor_id} - Datos procesados.")
            
            # Dibujar tras cada mensaje (o podrías añadir un contador para no saturar)
            render_plots()
            
    except KeyboardInterrupt:
        print("\nConsumo detenido por el usuario.")
    finally:
        consumer.close()

In [31]:
def process_message(message_value):
    """Extrae y almacena los datos del sensor."""
    s_id = message_value['producer_id']
    
    # Inicialización con deque para manejo automático de tamaño
    if s_id not in data_by_sensor:
        data_by_sensor[s_id] = {
            "timestamps": deque(maxlen=MAX_SAMPLES),
            **{k: deque(maxlen=MAX_SAMPLES) for k in METRICS.keys()}
        }
    
    storage = data_by_sensor[s_id]
    storage["timestamps"].append(message_value['timestamp'])
    for key in METRICS:
        storage[key].append(message_value[key])
    
    return s_id

In [32]:
# Kafka configuration
def initialize_consumer():
    kafka_topic = "water_quality"
    kafka_bootstrap_servers = ["localhost:9092"]

    # Create Kafka consumer
    consumer = KafkaConsumer(
        kafka_topic,
        bootstrap_servers=kafka_bootstrap_servers,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='latest',
        enable_auto_commit=True,
        group_id="water_consumers_group",   # Aniadimos el consumidor a un grupo
        key_deserializer=lambda k: k.decode('utf-8') if k else None
    )
    return consumer

In [ ]:
import matplotlib.dates as mdates
consumer = initialize_consumer()
run_consumer(consumer)

Iniciando consumo de mensajes...
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_1 - Datos procesados.
Sensor ID: producerID_4 - Datos procesados.
Sensor ID: producerID_5 - Datos procesados.
Sensor ID: producerID_6 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_0 - Datos procesados.
Sensor ID: producerID_2 - Datos procesados.
Sensor ID: producerID_3 - Datos procesados.